# 🍯🌡️ The viscous background, measured at three temperatures — HB vs TC on Carbopol in glycerin

*The same workflow as the [rheofit case study](https://rheofit.readthedocs.io/en/latest/walkthrough-carbopol-glycerin.html), run live in JupyterLite. Three flow curves of Carbopol in glycerin (20/30/40 °C) are fit with **Herschel–Bulkley** and the **three-component (TC)** model; TC's background viscosity η_bg(T) is then held up against the literature Arrhenius law for pure glycerol.*

**Run each cell in order** (▶). The first cell installs `rheofit` in your browser — no server involved.

In [ ]:
%pip install -q rheofit rheopy-rheodata


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rheofit
import rheodata
print("rheofit", rheofit.__version__)
print("rheodata", getattr(rheodata, "__version__", "installed"))


## 🧪 The dataset

The [rheodata](https://github.com/rheopy/rheodata) library (`caggioni_carbopol_glycerin_temp`) holds equilibrium flow curves of Carbopol in glycerin — 51 points each, 0.001 to 100 s⁻¹ — at 20, 30 and 40 °C (the 50 °C sweep is excluded, just like the published case study).

In [ ]:
# three temperature sweeps from rheodata (the 50 °C sweep is excluded, as in the case study)
dfs = {}
for T, sid in [(40, "T_40"), (30, "T_30"), (20, "T_20")]:
    d = rheodata.to_rheofit("caggioni_carbopol_glycerin_temp", sid)
    print(f"{T} °C: {len(d)} points")
    dfs[T] = d

x = dfs[20]["Shear rate / 1/s"].to_numpy()   # same shear-rate grid at every T
y = {T: dfs[T]["Stress / Pa"].to_numpy() for T in dfs}


## 🥊 Head-to-head: HB vs TC at every temperature

Six fits — HB and TC at 20, 30 and 40 °C, `effort="thorough"`, `seed=0`. ⏳ *This takes a few minutes in the browser.*

In [ ]:
fits = {}
for T in [40, 30, 20]:
    hb = rheofit.fit(dfs[T], "herschel_bulkley", effort="thorough", seed=0)
    tc = rheofit.fit(dfs[T], "tc", effort="thorough", seed=0)
    fits[T] = (hb, tc)
    print(f"T={T} °C  HB RedChi2={hb['redchi']:.3e}  TC RedChi2={tc['redchi']:.3e}")

print()
print(f"{'T (°C)':>6} | {'HB RedChi²':>10} | {'TC RedChi²':>10} | "
      f"{'σ_y / Pa':>12} | {'γ̇_c / 1/s':>12} | {'η_bg / Pa·s':>12}")
print("-" * 82)
for T in [40, 30, 20]:
    hb, tc = fits[T]
    p = {k: v for k, v in tc["params"].items()}
    def pv(name):
        v = p[name]
        return "%8.4g ± %4.1f%%" % (v["value"], 100*v["stderr"]/v["value"])
    print(f"{T:>6} | {hb['redchi']:>10.2e} | {tc['redchi']:>10.2e} | "
          f"{pv('sigma_y'):>12} | {pv('gamma_dot_c'):>12} | {pv('eta_bg'):>12}")

## 📉 Flow curves with TC fits

In [ ]:
def tc_stress(g, sigma_y, gamma_dot_c, eta_bg):
    return sigma_y + sigma_y * np.sqrt(g / gamma_dot_c) + eta_bg * g

colors = {40: "#d7301f", 30: "#fd8d3c", 20: "#2171b5"}
g = np.logspace(-3, 2, 400)

fig, ax = plt.subplots(figsize=(7, 5))
for T in [40, 30, 20]:
    p = {k: v["value"] for k, v in fits[T][1]["params"].items()}
    ax.loglog(x, y[T], "o", ms=4, color=colors[T], label=f"data ({T} °C)")
    ax.loglog(g, tc_stress(g, p["sigma_y"], p["gamma_dot_c"], p["eta_bg"]),
              "-", lw=2, color=colors[T],
              label=f"TC (RedChi²={fits[T][1]['redchi']:.2e})")
ax.set_xlabel("shear rate / 1/s")
ax.set_ylabel("stress / Pa")
ax.set_title("Carbopol in glycerin: TC fits at 20/30/40 °C")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

## 📈 Every TC parameter trends the physical way

- **σ_y grows on cooling** (5.36 → 7.49 Pa): the microgel network strengthens.
- **γ̇_c falls on cooling** (0.0156 → 0.0056 s⁻¹): the plastic √γ̇ term takes over at progressively lower shear rates as the background thickens.
- **η_bg thickens on cooling** (0.66 → 2.17 Pa·s): the background viscosity itself is strongly temperature-dependent — exactly what you expect if it is the solvent.

In [ ]:
Ts = np.array([40, 30, 20])
tc_p = {name: np.array([fits[T][1]["params"][name]["value"] for T in Ts])
        for name in ["sigma_y", "gamma_dot_c", "eta_bg"]}

fig, axs = plt.subplots(1, 3, figsize=(11, 3.6), sharex=True)
for ax, name, lab in zip(axs, ["sigma_y", "gamma_dot_c", "eta_bg"],
        ["σ_y / Pa", "γ̇_c / 1/s", "η_bg / Pa·s"]):
    ax.semilogy(Ts, tc_p[name], "o-", color="#2171b5", lw=2)
    ax.set_xlabel("temperature / °C")
    ax.set_ylabel(lab)
    ax.grid(True, which="both", alpha=0.3)
fig.suptitle("TC parameters vs temperature")
plt.tight_layout(); plt.show()

## 🌡️ The Arrhenius test

If η_bg really is the continuous phase (plus whatever the microgel adds at high shear), it should follow the solvent's temperature law: ln η = ln A + E_a/(R·T). Literature glycerol data (Segur & Oberstar 1951: 1.412, 0.612, 0.284 Pa·s at 20/30/40 °C) give E_a = 61.2 kJ/mol — a textbook straight line.

In [ ]:
R = 8.314  # J/mol/K
TK = Ts + 273.15
eta_bg = tc_p["eta_bg"]
eta_gly = np.array([0.284, 0.612, 1.412])  # literature glycerol, 40/30/20 °C

invT = 1.0 / TK
m_bg, b_bg = np.polyfit(invT, np.log(eta_bg), 1)
Ea_bg = m_bg * R / 1000.0  # kJ/mol
yfit = m_bg * invT + b_bg
ss_res = np.sum((np.log(eta_bg) - yfit) ** 2)
ss_tot = np.sum((np.log(eta_bg) - np.mean(np.log(eta_bg))) ** 2)
R2 = 1 - ss_res / ss_tot
print(f"TC η_bg:    E_a = {Ea_bg:.1f} kJ/mol, R² = {R2:.4f}")
print(f"literature: E_a = 61.2 kJ/mol, R² = 1.0000")
print("η_bg / η_glycerol:", ", ".join(f"{a/b:.2f}" for a, b in zip(eta_bg, eta_gly)))

invTg = np.linspace(invT.min(), invT.max(), 100)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(1e3 * invT, np.log(eta_bg), "o", ms=8, color="#2171b5",
        label=f"TC η_bg (E_a={Ea_bg:.1f} kJ/mol)")
ax.plot(1e3 * invTg, m_bg * invTg + b_bg, "-", lw=2, color="#2171b5")
m_gly, b_gly = np.polyfit(invT, np.log(eta_gly), 1)
ax.plot(1e3 * invT, np.log(eta_gly), "s", ms=8, color="#d7301f",
        label="literature glycerol (E_a=61.2 kJ/mol)")
ax.plot(1e3 * invTg, m_gly * invTg + b_gly, "--", lw=2, color="#d7301f")
ax.set_xlabel("1000 / T / 1/K")
ax.set_ylabel("ln(η / Pa·s)")
ax.set_title("Arrhenius test: TC η_bg vs literature glycerol")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout(); plt.show()

## 💡 What this tells us

Two things to read off the Arrhenius plot:

1. **η_bg tracks the solvent, always above it.** η_bg/η_glycerol runs 2.33 (40 °C) → 1.73 → 1.54 (20 °C): the background is glycerin *plus* the Carbopol microgel's own high-shear contribution.
2. **The slope is weaker (45.3 vs 61.2 kJ/mol) — and that makes sense.** The microgel contribution is only weakly temperature-dependent, so it dilutes the solvent's steep Arrhenius slope. Cold, the thick solvent dominates the background; hot, the solvent has thinned fivefold and the microgel carries a larger share — hence the growing ratio.

**Takeaways:**

- **TC beats HB 3–4× on RedChi² at every temperature** — when the continuous phase is viscous, the explicit η_bg term is not a luxury, it is the model.
- **η_bg(T) is Arrhenius with E_a ≈ 45 kJ/mol**, weaker than pure glycerol's 61.2 — the microgel's own high-shear contribution dilutes the solvent slope.
- **HB's n ≈ 0.52 is T-independent** — the microstructure's shear-thinning signature — while TC separates the physics instead of smearing it.